# 05 — Model Benchmarking, Training, Evaluation, and Registry

## Why Three Automation Layers?
- **LazyPredict**: fast breadth scan of many baseline estimators.
- **FLAML**: cost-aware AutoML optimization under time budget.
- **PyCaret**: higher-level ML workflow automation and tuning.

## Strengths / Weaknesses / Tradeoffs
- LazyPredict: fastest breadth, weak depth.
- FLAML: efficient optimization, depends on search budget and estimator support.
- PyCaret: very productive pipeline API, heavier dependency stack and version sensitivity.

## Theory
Champion model chosen by primary metric (RMSE), tie-breakers from R²/MAE.

In [ ]:
from modules.data_generator import run_data_augmentation
from modules.feature_engineering import build_feature_pipeline
from modules.feature_selector import run_feature_selection
from modules.model_trainer import run_training_pipeline
from modules.model_evaluator import evaluate_model
from modules.model_registry import register_model
from modules.settings import load_config, resolve_path

cfg = load_config()
aug = run_data_augmentation(cfg)
feat = build_feature_pipeline(aug, cfg)
ranking = run_feature_selection(feat, cfg)
selected = ranking.head(15)['feature'].tolist()

training = run_training_pipeline(feat, selected_features=selected, config=cfg)
training['leaderboard'].head(10)

In [ ]:
eval_result = evaluate_model(
    model=training['model'],
    X_test=training['X_test'],
    y_test=training['y_test'],
    figures_dir=resolve_path(cfg, 'figures_dir'),
    prefix='champion'
)

registry = register_model(
    model=training['model'],
    model_name='screentime_predictor',
    base_dir=resolve_path(cfg, 'model_registry_dir'),
    metrics=eval_result['metrics'],
    hyperparameters={'model_name': training['model_name'], 'source': training['model_source']},
    feature_names=training['feature_columns'],
    stage='staging',
    mlflow_run_id=training.get('mlflow_run_id'),
)

print('Champion:', training['model_name'], '| source:', training['model_source'])
print('Metrics:', eval_result['metrics'])
print('Registry:', registry)

In [ ]:
training['candidate_scoreboard']

## Metric Explanation
- MAE: average absolute error magnitude.
- MSE/RMSE: penalize larger errors strongly.
- R²: variance explained.
- MAPE: relative error percentage.

## Common Mistakes
- selecting champion on train metrics
- no holdout residual diagnostics
- no model version metadata